# CLMBR smoke test (Stage 0)

Validates that the official frozen CLMBR encoder (`StanfordShahLab/clmbr-t-base`) loads and runs
via the `femr` library on this cluster. Executed headless on a **public GPU node** (see
`jobs/clmbr_smoke.sbatch`), inside a `$SCRATCH` venv. Success = a `representations` tensor prints.

Research environment only — not a clinical tool; no causal-validity claims.

In [ ]:
# Cell 1 — pinned install (runs inside the active $SCRATCH venv)
!pip install torch==2.1.2 femr==0.2.3 datasets==2.15.0 xformers transformers==4.35.2

In [ ]:
import femr.models.transformer
import torch
import femr.models.tokenizer
import femr.models.processor
import datetime

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

model_name = "StanfordShahLab/clmbr-t-base"

# Load tokenizer / batch loader
tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(model_name)
batch_processor = femr.models.processor.FEMRBatchProcessor(tokenizer)

# Load model
model = femr.models.transformer.FEMRModel.from_pretrained(model_name)
print('model loaded:', type(model).__name__)

In [ ]:
# Create an example patient to run inference on
# This patient follows the MEDS schema: https://github.com/Medical-Event-Data-Standard
example_patient = {
    'patient_id': 30,
    'events': [{
        'time': datetime.datetime(2011, 5, 8),
        'measurements': [
            {'code': 'SNOMED/184099003'},
            {'code': 'Visit/IP'},
        ],
    },
    {
        'time': datetime.datetime(2012, 6, 9),
        'measurements': [
            {'code': 'Visit/OP'},
            {'code': 'SNOMED/3950001'}
        ],
    }]
}

raw_batch = batch_processor.convert_patient(example_patient, tensor_type="pt")
batch = batch_processor.collate([raw_batch])

# Run model
with torch.no_grad():
    _, result = model(**batch)
    print(result['timestamps'].cpu().numpy().astype('datetime64[s]'))
    print(result['patient_ids'])
    print('representations shape:', tuple(result['representations'].shape))
    print(result['representations'])